# CSMI Analysis

Central Sensori-Motor Integration (CSMI) analysis using the Peterka18 model.
Performs model fits and plots analysis results for the prts stimulus condition.

Results are saved to `../results/csmi_results.parquet`.

In [1]:
import numpy as np
import pandas as pd
import balancepy as bp
from joblib import Parallel, delayed
from pathlib import Path
import sys; sys.path.insert(0, '..')
from src.config import paths

# Configuration for preprocessing and SRData

In [2]:
# preprocessing configuration
preproc_config = bp.AnaropiaPreprocessingConfig()
preproc_config.end_time_seconds = 300
preproc_config.cut_to_cycles = True
preproc_config.cycle_length_samples = 1800
preproc_config.cycle_start_samples = 1800
print(preproc_config)

# stimulus-response class configuration
sr_config = bp.AnaropiaSRDataConfig(
    stimulus_name='stim_pitch', 
    response_name=bp.COM_LEGACY_AP, 
    frequency_selection='prts'
    )
print(sr_config)

AnaropiaPreprocessingConfig(
  samplingrate_Hz: 90
  resample: True
  end_time_seconds: 300
  filter_type: None
  filter_order: 2
  filter_cutoff_Hz: 8
  cut_to_cycles: True
  cycle_start_samples: 1800
  cycle_length_samples: 1800
)
AnaropiaSRDataConfig(stimulus_name='stim_pitch', response_name=COMConfig(shoulder_pos_column='shld_zpos', hip_pos_column='hip_zpos', shoulder_height_column='shld_ypos', hip_height_column='hip_ypos', rotation=True), frequency_selection='prts', name=None, column_scales=None)


## Load metadata

In [3]:
# run syncronization with quality flag file
metadata = bp.get_metadata(paths=paths)

## Example — single subject CSMI analysis

`bp.run_csmi` returns a fitted `Peterka18` object with access to parameters, fit quality, and plots.

In [4]:
subject_id = 'KA07RI18'
filename = bp.get_filename_from_metadata(paths, subject_id, 'c12: prts', metadata=metadata)
body_height_m = metadata.loc[metadata['Subject ID'] == subject_id, 'body_height_m'].values[0]
body_weight_kg = metadata.loc[metadata['Subject ID'] == subject_id, 'body_weight_kg'].values[0]

subj = bp.run_csmi(filename, body_height_m=body_height_m, body_weight_kg=body_weight_kg, preproc_config=preproc_config, sr_config=sr_config)
subj.plot()

## Batch CSMI analysis

Fits the Peterka18 model for all subjects in parallel. Fitted parameter columns follow the naming
convention `{param_name}_{condition_slug}` (e.g. `W_c12_prts`).

In [5]:
if True:  # set to True to run CSMI batch processing
    jobs = []
    for subject_id, condition, filepath, body_height_m, body_weight_kg in bp.batch_iterator(
        paths,
        conditions=['c12: prts'],
        output_columns=['body_height_m', 'body_weight_kg'],
        skip_flags=None,
    ):
        condition_slug = condition.replace(': ', '_').replace(' ', '_')
        plot_file = paths.csmi_plots / f"csmi_{subject_id}_{condition_slug}.png"

        if not plot_file.exists():
            jobs.append((filepath, body_height_m, body_weight_kg, subject_id, condition_slug, plot_file))

    paths.csmi_plots.mkdir(parents=True, exist_ok=True)

    def _run_csmi_and_plot(job):
        filepath, body_height_m, body_weight_kg, subject_id, condition_slug, plot_file = job

        subj = bp.run_csmi(
            filepath,
            body_height_m=body_height_m,
            body_weight_kg=body_weight_kg,
            sr_config=sr_config,
            preproc_config=preproc_config,
            name=f"{subject_id}_{condition_slug}",
        )

        fig = subj.plot()
        if fig is not None:
            fig.write_image(str(plot_file))

        return subj

    subjects = Parallel(n_jobs=3)(
        delayed(_run_csmi_and_plot)(job) for job in jobs
    )